In [5]:
from pyspark.sql import functions as F
from functools import reduce

# file name -> (indicator code, file encoding)
files = {
    "central_government_debt.csv":      ("CGD", "iso-8859-1"),
    "general_government_debt.csv":      ("GGD", "UTF-8"),
    "household_debt.csv":               ("HHD", "iso-8859-1"),
    "non-financial_corporate_debt.csv": ("NFC", "iso-8859-1"),
    "private_debt.csv":                 ("PVD", "iso-8859-1"),
}

StatementMeta(, e31fb116-1af2-401c-b113-01d9b0699a2f, 7, Finished, Available, Finished, False)

In [2]:
def load_file(name, code, encoding):
    df = (spark.read
            .option("header", True)
            .option("encoding", encoding)
            .csv(f"Files/bronze/{name}"))

    # Clean column names: remove hidden BOM characters and spaces
    for c in df.columns:
        df = df.withColumnRenamed(c, c.replace("\ufeff", "").strip())

    year_cols = [c for c in df.columns if c.isdigit()]

    return (df.unpivot(ids=["country_name", "indicator_name"],
                       values=year_cols,
                       variableColumnName="year",
                       valueColumnName="value")
              .withColumn("country_name", F.trim("country_name"))
              .withColumn("year", F.col("year").cast("int"))
              .withColumn("debt_pct_gdp", F.col("value").cast("double"))
              .drop("value")
              .filter(F.col("debt_pct_gdp").isNotNull())
              .withColumn("indicator_code", F.lit(code))
              .withColumn("source_file", F.lit(name)))

StatementMeta(, e31fb116-1af2-401c-b113-01d9b0699a2f, 4, Finished, Available, Finished, False)

In [6]:
parts = [load_file(f, code, enc) for f, (code, enc) in files.items()]
silver = reduce(lambda a, b: a.unionByName(b), parts)

silver.write.mode("overwrite").format("delta").saveAsTable("silver.debt_long")
print("Saved silver.debt_long")

StatementMeta(, e31fb116-1af2-401c-b113-01d9b0699a2f, 8, Finished, Available, Finished, False)

Saved silver.debt_long


In [7]:
df = spark.table("silver.debt_long")
print("Total rows:", df.count())
print("Countries:", df.select("country_name").distinct().count())
display(df.groupBy("indicator_code").count().orderBy("indicator_code"))

StatementMeta(, e31fb116-1af2-401c-b113-01d9b0699a2f, 9, Finished, Available, Finished, False)

Total rows: 24399
Countries: 189


SynapseWidget(Synapse.DataFrame, 8597c47f-c2b2-4838-87a7-a103c49d289e)

In [8]:
silver = spark.table("silver.debt_long")

regions = (spark.read.option("header", True)
             .option("encoding", "UTF-8")
             .csv("Files/bronze/country_region.csv"))

dim_country = (silver.select("country_name").distinct()
                 .join(regions, "country_name", "left")
                 .withColumn("is_nigeria", F.col("country_name") == "Nigeria"))

print("Countries without a region:", dim_country.filter(F.col("region").isNull()).count())
dim_country.write.mode("overwrite").format("delta").saveAsTable("gold.dim_country")

StatementMeta(, e31fb116-1af2-401c-b113-01d9b0699a2f, 10, Finished, Available, Finished, False)

Countries without a region: 0


In [9]:
ind_rows = [
    ("CGD", "Central Government Debt",      "Public",  1),
    ("GGD", "General Government Debt",      "Public",  2),
    ("PVD", "Private Debt",                 "Private", 3),
    ("HHD", "Household Debt",               "Private", 4),
    ("NFC", "Non-financial Corporate Debt", "Private", 5),
]
dim_indicator = spark.createDataFrame(ind_rows, ["indicator_code", "indicator", "sector", "sort_order"])
dim_indicator.write.mode("overwrite").format("delta").saveAsTable("gold.dim_indicator")

StatementMeta(, e31fb116-1af2-401c-b113-01d9b0699a2f, 11, Finished, Available, Finished, False)

In [10]:
fact = silver.select("country_name", "indicator_code", "year", "debt_pct_gdp")
fact.write.mode("overwrite").format("delta").saveAsTable("gold.fact_debt")

dim_year = (spark.range(1950, 2023).withColumnRenamed("id", "year")
              .withColumn("year", F.col("year").cast("int"))
              .withColumn("decade", (F.floor(F.col("year") / 10) * 10).cast("int")))
dim_year.write.mode("overwrite").format("delta").saveAsTable("gold.dim_year")

StatementMeta(, e31fb116-1af2-401c-b113-01d9b0699a2f, 12, Finished, Available, Finished, False)

In [11]:
for t in ["gold.dim_country", "gold.dim_indicator", "gold.dim_year", "gold.fact_debt"]:
    print(t, spark.table(t).count())

display(spark.table("gold.dim_country").groupBy("region").count().orderBy("region"))
display(spark.table("gold.dim_country").filter(F.col("country_name").rlike("[^ -~]")))

StatementMeta(, e31fb116-1af2-401c-b113-01d9b0699a2f, 13, Finished, Available, Finished, False)

gold.dim_country 189
gold.dim_indicator 5
gold.dim_year 73
gold.fact_debt 24399


SynapseWidget(Synapse.DataFrame, 17a1e094-4d97-4536-9315-cf07541c7c53)

SynapseWidget(Synapse.DataFrame, 50f7e071-7508-4e28-b6e2-fc6b004e8866)